# 第五阶段：智能指针

## 实验 1：复现 raw pointer 的 ownership 问题

本实验不以熟练使用 `new/delete` 为目标，而是观察手动资源管理为什么脆弱：

- `new` 创建的对象不会随 raw pointer 离开作用域自动销毁；
- 每条控制流都必须由程序员手动执行一次 `delete`；
- 提前返回会轻易跳过清理，造成资源泄漏；
- raw pointer 的类型本身没有说明谁拥有对象。

请在重启 `xcpp23` kernel 后从上到下运行，避免上一次故意泄漏的对象影响存活计数。

In [1]:
// 本步骤：引入断言和输出工具，用于验证对象是否按预期销毁。
#include <cassert>
#include <iostream>

### 1. 用日志观察对象生命周期

`User` 用静态计数器记录当前仍存活的对象数。构造使计数加一，析构使计数减一，因此即使无法直接查看堆内存，也能观察某个对象是否完成了清理。

In [2]:
// 本步骤：定义带生命周期日志的对象，让构造、析构和泄漏都可观察。
class User
{
public:
    explicit User(int id)
        : id_(id)
    {
        // 构造完成后增加存活计数，表示堆上已经存在一个 User。
        ++live_count_;
        std::cout << "User " << id_ << " created, live = " << live_count_ << '\n';
    }

    ~User() noexcept
    {
        // 析构时减少计数；若没有看到这条日志，说明清理路径没有执行。
        --live_count_;
        std::cout << "User " << id_ << " destroyed, live = " << live_count_ << '\n';
    }

    static int live_count() noexcept
    {
        // 只读返回当前存活数量，供实验断言使用。
        return live_count_;
    }

private:
    int id_;
    inline static int live_count_ = 0;
};

### 2. 正常路径依赖手动 `delete`

`new User` 返回的 raw pointer 只是地址。编译器不会从 `User*` 判断它是 owner、borrower，还是可能为空的观察者。这里由局部变量 `user` 暂时承担 ownership，并要求函数在离开前手动释放对象。

In [3]:
// 本步骤：实现手动管理版本，并故意保留一条跳过 delete 的提前返回路径。
void process_raw(bool early_return)
{
    // new 在堆上创建对象；此刻 user 被人为约定为唯一 owner。
    User *user = new User(1);

    // 提前返回直接离开函数，raw pointer 被丢弃，但堆对象不会自动析构。
    if (early_return)
    {
        std::cout << "early return: delete is skipped\n";
        return;
    }

    // 只有执行到正常路径末尾时，程序才会释放对象。
    delete user;

    // delete 后地址已经失效；置空可避免本变量被误用，但不能解决遗漏 delete。
    user = nullptr;
}

In [4]:
// 本步骤：运行正常路径，确认 new 与 delete 成对时没有遗留对象。
{
    const int before = User::live_count();

    // false 让控制流执行函数末尾的 delete。
    process_raw(false);

    // 存活数回到调用前，说明本次创建的对象已经析构。
    assert(User::live_count() == before);
}

User 1 created, live = 1
User 1 destroyed, live = 0


预期输出同时出现 `created` 和 `destroyed`。这并不说明接口安全，只说明当前这条控制流恰好记得执行了 `delete`。

### 3. 提前返回造成泄漏

下一步会故意泄漏一个很小的教学对象。函数返回后，唯一的地址随局部变量消失，程序已无法再对该对象执行 `delete`。请观察只有构造日志，没有对应析构日志。

In [5]:
// 本步骤：触发提前返回，并用存活计数确认有一个对象没有被释放。
{
    const int before = User::live_count();

    // true 使函数在 delete 之前返回，所有权信息随 raw pointer 一起丢失。
    process_raw(true);

    // 比调用前多一个存活对象，证明析构没有发生。
    assert(User::live_count() == before + 1);
    std::cout << "leaked objects in this run = 1\n";
}

User 1 created, live = 1
early return: delete is skipped
leaked objects in this run = 1


### 4. 修补控制流不能改变脆弱的 ownership 模型

可以在当前 `return` 前补一个 `delete`，但函数以后加入异常、更多返回分支或 ownership 转移时，每条路径都必须继续保持完全配对。其他常见错误也来自同一个问题：

```cpp
delete user;
delete user;        // double free：同一对象释放两次，行为未定义

delete user;
user->hello();      // use-after-free：访问生命周期已结束的对象
```

这些失败代码只用于阅读，不在 Notebook 中执行。它们不是可靠的运行时错误，可能崩溃，也可能暂时看似正常。

### 5. 用内存诊断工具验证真实泄漏

仓库中提供了可独立编译的 [完整复现源码](./src/01_raw_pointer_problem.cpp)。macOS 可从仓库根目录运行：

```bash
clang++ -std=c++23 -g -O1 \
    basics/01-cpp/05-smart-pointer/src/01_raw_pointer_problem.cpp \
    -o /tmp/01_raw_pointer_problem
leaks --atExit -- /tmp/01_raw_pointer_problem
```

Linux 上可在编译和运行时分别加入 `-fsanitize=address` 与 `ASAN_OPTIONS=detect_leaks=1`，让 LeakSanitizer 报告泄漏。Apple AddressSanitizer 不支持所有平台上的 leak detection，因此这里优先使用 macOS 自带的 `leaks`。预期工具报告一个直接泄漏；Notebook 中的计数器用于解释现象，内存诊断工具才负责检查真实分配。

## 实验结论

```text
new 创建 User
      │
      ▼
User* 只保存地址
      │ ownership 规则只存在于人的约定中
      ├── 正常路径执行 delete ──> User 析构
      └── 提前返回跳过 delete ──> 泄漏
```

本实验中的 owner 是 `process_raw()`，对象由 `new` 创建，理应由同一 ownership 路径执行 `delete`。函数没有把 pointer 借给外部；提前返回时 pointer 失效不是问题，真正的问题是它指向的对象仍存活却再也无法释放。

raw pointer 适合表达非拥有观察或与底层 API 交互，但 `User*` 本身无法表达所有权。下一实验将用 `std::unique_ptr<User>` 把“唯一拥有并在作用域结束时自动释放”写进类型，让提前返回也能触发清理。智能指针是 C++ 内部的 RAII 类型；未来跨 C ABI 时仍需转换为 opaque handle 等 C 兼容表示。